# 02 · Features and A/B split — SepsisWatch Z

**Why:** the model does not understand "the patient is getting worse"; it only sees numbers per row.
This notebook turns each **hour of each patient** into a row of features that summarize their history **up to that hour**:
last measured value, whether it was measured, how long ago, 6-hour trends and clinical criteria.

**Golden rule:** each row only uses information from **that hour or earlier**. Never data from the future (in real life, at 10 AM you do not know what will happen at 11).

**Input:** `~/shared/sepsis/data/processed/sepsis_all.pkl` (from `01_ingest_exploration`)
**Output:** `~/shared/sepsis/data/processed/features.pkl` + `feature_list.json`
→ used by `03_xgboost` and `04_nnpa_mlp`.

## 1. Load the consolidated data

In [ ]:
import os, json, time, platform
import numpy as np
import pandas as pd

BASE = os.path.expanduser("~/shared/sepsis/data/processed")
t_start = time.time()
df = pd.read_pickle(f"{BASE}/sepsis_all.pkl")
df = df.sort_values(["patient_id", "hour"], kind="stable").reset_index(drop=True)
print(f"{len(df):,} rows | {df['patient_id'].nunique():,} patients | {platform.machine()}")

VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN",
        "Alkalinephos", "Calcium", "Chloride", "Creatinine", "Bilirubin_direct",
        "Glucose", "Lactate", "Magnesium", "Phosphate", "Potassium",
        "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC",
        "Fibrinogen", "Platelets"]
MEASURED = VITALS + LABS
g = df.groupby("patient_id", observed=True, sort=False)

## 2. Was it measured this hour? How many hours since the last measurement?

- `<var>_measured` = 1 if that variable was measured **this hour**. A clinician ordering a test is already information.
- `<var>_hours_since` = hours since the last measurement (empty if never measured).

In [ ]:
t0 = time.time()
feats = {}
measured_hour = pd.DataFrame(index=df.index)
for c in MEASURED:
    m = df[c].notna()
    feats[f"{c}_measured"] = m.astype(np.int8)
    measured_hour[c] = df["hour"].where(m).astype(np.float32)
last_hour = measured_hour.groupby(df["patient_id"], observed=True, sort=False).ffill()
for c in MEASURED:
    feats[f"{c}_hours_since"] = (df["hour"] - last_hour[c]).astype(np.float32)
del measured_hour, last_hour
print(f"{2 * len(MEASURED)} columns in {time.time() - t0:.1f} s")

## 3. Carry the last measured value forward (*forward fill*)

If lactate was measured at 3 AM, that value is kept until the next measurement.
This is done **within each patient** (a value is never carried from one patient to another).

In [ ]:
t0 = time.time()
filled = g[MEASURED].ffill().astype(np.float32)
for c in MEASURED:
    feats[c] = filled[c]
print(f"{len(MEASURED)} columns in {time.time() - t0:.1f} s")

## 4. 6-hour trends in vital signs

For heart rate, O2 saturation, temperature, systolic BP, mean arterial pressure and respiratory rate:
**mean, min and max** over the last 6 hours, and the **change** versus 6 hours ago (is it going up or down?).
The exploration showed that **the trend matters more than a single value**.

In [ ]:
t0 = time.time()
TREND = ["HR", "O2Sat", "Temp", "SBP", "MAP", "Resp"]
roll = filled[TREND].groupby(df["patient_id"], observed=True, sort=False).rolling(6, min_periods=1)
for stat in ["mean", "min", "max"]:
    r = getattr(roll, stat)().reset_index(level=0, drop=True).sort_index()
    for c in TREND:
        feats[f"{c}_6h_{stat}"] = r[c].astype(np.float32)
shifted = filled[TREND].groupby(df["patient_id"], observed=True, sort=False).shift(6)
for c in TREND:
    feats[f"{c}_6h_change"] = (filled[c] - shifted[c]).astype(np.float32)
del roll, shifted
print(f"{4 * len(TREND)} columns in {time.time() - t0:.1f} s")

## 5. Clinical variables (what a clinician looks at)

- **Shock index** = heart rate / systolic BP (> 1 is a bad sign).
- **SIRS** (0–4): temperature > 38 or < 36, heart rate > 90, respiratory rate > 20 or PaCO2 < 32, WBC > 12 or < 4.
- **qSOFA** (0–2 with the available data): respiratory rate ≥ 22, systolic BP ≤ 100. *(The third criterion, altered mental status, is not in the dataset.)*
- **Organ dysfunction:** lactate > 2, platelets < 150, creatinine > 1.2, bilirubin > 1.2.

SIRS and qSOFA are **current practice**: later we use them as the **baseline** to show our model does better.

In [ ]:
f = filled
feats["shock_index"] = (f["HR"] / f["SBP"]).astype(np.float32)
sirs = ((f["Temp"] > 38) | (f["Temp"] < 36)).astype(np.int8) \
     + (f["HR"] > 90).astype(np.int8) \
     + ((f["Resp"] > 20) | (f["PaCO2"] < 32)).astype(np.int8) \
     + ((f["WBC"] > 12) | (f["WBC"] < 4)).astype(np.int8)
feats["SIRS"] = sirs
feats["qSOFA"] = ((f["Resp"] >= 22).astype(np.int8) + (f["SBP"] <= 100).astype(np.int8))
feats["high_lactate"] = (f["Lactate"] > 2).astype(np.int8)
feats["low_platelets"] = (f["Platelets"] < 150).astype(np.int8)
feats["high_creatinine"] = (f["Creatinine"] > 1.2).astype(np.int8)
feats["high_bilirubin"] = (f["Bilirubin_total"] > 1.2).astype(np.int8)
feats["BUN_creatinine_ratio"] = (f["BUN"] / f["Creatinine"]).astype(np.float32)

# Patient fields
feats["Age"] = df["Age"]
feats["Gender"] = df["Gender"]
feats["Unit1"] = df["Unit1"].fillna(-1)
feats["Unit2"] = df["Unit2"].fillna(-1)
feats["HospAdmTime"] = df["HospAdmTime"]
feats["ICULOS"] = df["ICULOS"]
del filled, f

## 6. Build the final matrix and split into train / validation / test

- **Hospital A → 80 % train, 20 % validation.** The split is **by patient** (all hours of a patient go to the same group). Splitting by row would let the model "know" the patient and look better than it is (*data leakage*).
- The split keeps the share of sepsis patients (stratified).
- **Hospital B → final test only.** The model never sees it during training: this shows it **works in a different hospital**.

In [ ]:
F = pd.DataFrame(feats)
F = F.astype({c: np.float32 for c in F.columns if F[c].dtype == np.float64})
FEATURES = list(F.columns)
X = pd.concat([df[["patient_id", "hospital", "hour"]], F, df[["SepsisLabel"]]], axis=1)
del feats, F

pat = X.groupby("patient_id", observed=True).agg(hospital=("hospital", "first"), sepsis=("SepsisLabel", "max"))
rng = np.random.default_rng(42)
split = pd.Series("test", index=pat.index)
for s in [0, 1]:
    ids = pat[(pat["hospital"] == "A") & (pat["sepsis"] == s)].index.to_numpy()
    rng.shuffle(ids)
    n_val = int(round(0.2 * len(ids)))
    split[ids[:n_val]] = "val"
    split[ids[n_val:]] = "train"
X["split"] = X["patient_id"].map(split).astype("category")

split_summary = X.groupby("split", observed=True).agg(
    patients=("patient_id", "nunique"), rows=("hour", "size"), pct_sepsis_hours=("SepsisLabel", "mean"))
split_summary["pct_sepsis_hours"] = (100 * split_summary["pct_sepsis_hours"]).round(2)
split_summary["patients_with_sepsis"] = X[X["SepsisLabel"] == 1].groupby("split", observed=True)["patient_id"].nunique()
print(f"Features: {len(FEATURES)} | Memory: {X.memory_usage(deep=True).sum() / 1024**2:,.0f} MB")
split_summary

## 7. Quality checks

1. No patient appears in two groups (no leakage between train, validation and test).
2. No feature uses the future: the first hour of each patient cannot have a 6-hour change.
3. Is there signal? We compare some features between non-sepsis hours (0) and sepsis hours (1).

In [ ]:
assert X.groupby("patient_id", observed=True)["split"].nunique().max() == 1, "Patient in two groups"
assert X.loc[X["hour"] == 0, "HR_6h_change"].notna().sum() == 0, "Hour 0 has a 6h change"
print("OK: no leakage between groups and no future data at hour 0")

check = ["HR", "Resp", "Temp", "MAP", "shock_index", "SIRS", "qSOFA", "HR_6h_change", "Lactate", "ICULOS"]
X[X["split"] == "train"].groupby("SepsisLabel")[check].mean().T.round(2)

## 8. Save for the models

- `features.pkl`: the full matrix (`float32`) with `patient_id`, `hospital`, `hour`, `split` and `SepsisLabel`.
- `feature_list.json`: the exact order of the model input columns (the `scorer` and the NNPA network must use the same order).

**Note for the neural network (`04`):** features contain empty values (`NaN`) when something was never measured.
XGBoost handles them natively; the network needs them filled (e.g. with the train median) and standardized, using train statistics only.

In [ ]:
X.to_pickle(f"{BASE}/features.pkl")
with open(f"{BASE}/feature_list.json", "w") as fh:
    json.dump(FEATURES, fh, indent=1)
print(f"Saved: {BASE}/features.pkl ({os.path.getsize(f'{BASE}/features.pkl') / 1024**2:,.0f} MB)")
print(f"Saved: {BASE}/feature_list.json ({len(FEATURES)} features)")
print(f"Total notebook time: {time.time() - t_start:.1f} s on {platform.machine()}")